# ZTE — run on Google Colab (GPU) or locally

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/victor-iyi/zte/blob/main/notebooks/zte_colab.ipynb)

This notebook runs the whole ZTE pipeline — including the **leave-one-subject-out (LOSO) “new brain” experiment** — with the accelerator chosen automatically (**CUDA on Colab, MPS on a Mac, CPU otherwise**).

Everything is **resumable**: stop any cell and re-run it, or re-run a script, and it continues exactly where it left off.

> ZTE requires **Python 3.14**, which Colab does not ship. We use [`uv`](https://docs.astral.sh/uv/) to provision Python 3.14 and install the project — no system Python changes needed.

## 1 · Set up (uv provisions Python 3.14 + installs ZTE)

In [ ]:
import os

!pip install -q uv
if not os.path.isdir('zte') and not os.path.isfile('pyproject.toml'):
    !git clone --depth 1 https://github.com/victor-iyi/zte.git
if os.path.isdir('zte'):
    %cd zte
# Provision Python 3.14 and install torch (CUDA on Colab GPU) + all extras. Cached across runs.
!uv python install 3.14
!uv sync --group all

## 2 · Confirm the accelerator (auto-selected)
On a Colab **GPU runtime** this prints a CUDA device; on CPU it says so. No flags needed — `--device auto` (the default) picks the best backend.

In [ ]:
!uv run python -c "import torch; from zte.device import resolve_device; s=resolve_device('auto'); print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| ZTE will use:', s.name)"

## 3 · Smoke test on synthetic data (no dataset needed)
Proves the whole pipeline trains + evaluates + writes the interactive explorers in a couple of minutes.

In [ ]:
!uv run zte-run --config experiments/exp6_skipgram_eegonly_invariant.yaml --synthetic --epochs 3 --name colab_smoke

## 4 · The LOSO “new brain” experiment (resumable)
Trains the invariance recipe once per held-out subject, turning the single-subject result into a **trend**. `SMOKE=1` runs a fast synthetic dry-run; drop it (and pass a data root, next section) for the real multi-hour run.

**Resume:** if the runtime disconnects, just run this cell again — finished subjects are skipped and the interrupted one continues from its last checkpoint.

In [ ]:
!SMOKE=1 bash scripts/run_loso.sh

## 5 · Real ZuCo data (from Google Drive)
Mount Drive and point the sweep at your extracted `.mat` folder (or download a Drive folder first). The GPU is used automatically.

Uncomment the cell below to run for real (this is the multi-hour run):

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
#
# Option A — you already have extracted .mat files on Drive:
# !bash scripts/run_loso.sh /content/drive/MyDrive/zuco_extracted
#
# Option B — download a shared Drive folder of ZuCo .zip archives first:
# !uv run zte-download --drive <FOLDER_ID_OR_URL> --extract-dir res/data/zuco_extracted
# !bash scripts/run_loso.sh res/data/zuco_extracted

## 6 · Combine everything into one interactive view
`zte-compare` reads every run and builds a single scorecard + best-run dashboard. Each run also has its own **Thought-Space Explorer** and **Neuron Atlas** under `res/experiments/<run>/evaluation/interactive/`.

In [ ]:
!uv run zte-compare --experiments res/experiments/loso --out res/experiments/loso/COMPARE.html --title 'ZTE — LOSO (new-brain) trend'
from IPython.display import IFrame

IFrame('res/experiments/loso/COMPARE.html', width='100%', height=720)